# FinTech Reconciliation and Normalization Workflow
This notebook loads raw reconciliation data, performs validation checks, creates reconciliation outputs, and normalizes the JSON API sample.


In [ ]:
import pandas as pd
import json
from pathlib import Path
root = Path("../01_data/raw")


In [ ]:
ledger_df = pd.read_csv(root / "ledger.csv")
gateway_df = pd.read_csv(root / "gateway.csv")
ledger_df.head()


In [ ]:
print("Ledger rows", len(ledger_df))
print("Gateway rows", len(gateway_df))
print("Ledger nulls:
", ledger_df.isna().sum())
print("Gateway nulls:
", gateway_df.isna().sum())


In [ ]:
merged = ledger_df.merge(gateway_df, on=["transaction_id","merchant_id"], how="outer", suffixes=("_ledger","_gateway"), indicator=True)
missing_in_gateway = merged[merged["_merge"]=="left_only"]
missing_in_ledger = merged[merged["_merge"]=="right_only"]
amount_mismatches = merged[(merged["_merge"]=="both") & (merged["amount_usd_ledger"] != merged["amount_usd_gateway"])]
status_mismatches = merged[(merged["_merge"]=="both") & (merged["status_ledger"].str.lower() != merged["status_gateway"].str.lower())]
missing_in_gateway.head(), missing_in_ledger.head(), amount_mismatches.head(), status_mismatches.head()


In [ ]:
report = merged.copy()
report["issue_type"] = "match"
report.loc[report["_merge"]=="left_only", "issue_type"] = "missing_in_gateway"
report.loc[report["_merge"]=="right_only", "issue_type"] = "missing_in_ledger"
report.loc[(report["_merge"]=="both") & (report["amount_usd_ledger"] != report["amount_usd_gateway"]), "issue_type"] = "amount_mismatch"
report.loc[(report["_merge"]=="both") & (report["status_ledger"].str.lower() != report["status_gateway"].str.lower()), "issue_type"] = "status_mismatch"
report.to_csv(root.parent / "processed" / "reconciliation_report.csv", index=False)
missing_in_gateway.to_csv(root.parent / "processed" / "missing_in_gateway.csv", index=False)
missing_in_ledger.to_csv(root.parent / "processed" / "missing_in_ledger.csv", index=False)
amount_mismatches.to_csv(root.parent / "processed" / "amount_mismatches.csv", index=False)
status_mismatches.to_csv(root.parent / "processed" / "status_mismatches.csv", index=False)
report.head()


In [ ]:
with open(root / "../raw" / "api_response_sample.json", "r", encoding="utf-8") as f:
    api = json.load(f)
rows = []
for batch in api.get("batches", []):
    merchant = batch.get("merchant", {})
    for settlement in batch.get("settlements", []):
        bank = settlement.get("bank", {})
        rows.append({
            "batch_id": batch.get("batch_id"),
            "merchant_id": merchant.get("merchant_id"),
            "merchant_name": merchant.get("merchant_name"),
            "merchant_region": merchant.get("region"),
            "settlement_id": settlement.get("settlement_id"),
            "amount_usd": settlement.get("amount_usd"),
            "status": settlement.get("status"),
            "processed_at": settlement.get("processed_at"),
            "bank_name": bank.get("name"),
            "bank_country": bank.get("country")
        })
api_df = pd.DataFrame(rows)
api_df["processed_at"] = pd.to_datetime(api_df["processed_at"], errors="coerce")
api_df.to_csv(root.parent / "processed" / "api_normalized.csv", index=False)
api_df.head()
